In [2]:
import sys
print(sys.executable)
import subprocess
result = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

c:\Users\yunju\anaconda3\envs\Quantization\python.exe
2.6.0+cpu




In [3]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import DataLoader

model_name = "facebook/opt-125m"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

data = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

def load_data(path: str, name: str):
    data = load_dataset(path, name)
    
    train_data = [x["text"] for x in data["train"] if x["text"] != ""]
    test_data = [x["text"] for x in data["test"] if x["text"] != ""]
    valid_data = [x["text"] for x in data["validation"] if x["text"] != ""]
    
    print("train length : ", len(train_data))
    print("test length : ", len(test_data))
    print("valid length : ", len(valid_data))     
    return train_data, test_data, valid_data

train_data, test_data, valid_data = load_data("Salesforce/wikitext", "wikitext-2-raw-v1")        

c:\Users\yunju\anaconda3\envs\Quantization\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 48216.02it/s]


train length :  23767
test length :  2891
valid length :  2461


In [4]:
def collate_fn(data):
    batch = tokenizer(
        data,
        return_tensors = "pt",
        padding = True,
        truncation = True,
        max_length = 128,
    )
    
    batch = {k: v.to(device) for k, v in batch.items()}
    return batch


train_loader = DataLoader(
    train_data,
    batch_size = 128,
    shuffle = True,
    collate_fn = collate_fn
)

In [6]:
import torch.nn as nn

class Quantizer(nn.ModuleList):
    def __init__(self, model, data_loader, bits = 8, block_size = 128):
        super(Quantizer, self).__init__()
        self.model = model
        self.data_loader = data_loader
        self.bits = bits
        self.block_size = block_size
    
    def collect_input_n_weights(self):
        acts = {}
        weights = {}
        handles = []

        for name, module in self.model.named_modules():
            if isinstance(module, nn.Linear):
                acts[name] = []
                weights[name] = module.weight.detach().float()


                def make_hook(n):
                    def hook_fn(module, inputs, output):
                        acts[n].append(inputs[0].detach().float().cpu())
                    return hook_fn
                
                handles.append(module.register_forward_hook(make_hook(name)))
        
        self.model.eval()
        with torch.no_grad():
            batch = next(iter(self.data_loader))
            self.model(**batch)

        for handle in handles:
            handle.remove()
        
        X = {name: torch.cat([x.reshape(-1, x.shape[-1]) for x in tensors], dim = 0)
             for name, tensors in acts.items()}        
        
        return X, weights
    
    def Hessian(self, X):
        H = X.T @ X
        H = 2 * H
        return H
    
    def gptq_quantize(self, w):
        w_min = w.min()
        w_max = w.max()
        scale = (w_max - w_min) / (2**self.bits - 1)
        zero_point = torch.round(-w_min / scale)
        
        w_q = torch.clamp(torch.round(w / scale) + zero_point, 0, 2**self.bits - 1)
        w_dq = (w_q - zero_point) * scale
        return w_q, w_dq

    
    def _gptq(self, W, H):
        out_features, in_features = W.shape
        W_q = torch.zeros_like(W)
        
        # Hessian damping (수치 안정성)
        damp = 0.01 * H.diag().mean()
        H += damp * torch.eye(in_features)
        
        # Cholesky 분해로 H_inv 계산
        L = torch.linalg.cholesky(H)
        H_inv = torch.cholesky_inverse(L)  # [in_features, in_features]
        
        # Block-wise quantization
        for col_start in range(0, in_features, self.block_size):
            col_end = min(col_start + self.block_size, in_features)
            
            W_block = W[:, col_start:col_end].clone()           # [out, block]
            W_q_block = torch.zeros_like(W_block)
            err_block = torch.zeros_like(W_block)
            H_inv_block = H_inv[col_start:col_end, col_start:col_end]  # [block, block]
            
            # Block 내부 column 순차 양자화
            for i in range(col_end - col_start):
                w = W_block[:, i]           # [out_features]
                h_inv_ii = H_inv_block[i, i]
                
                w_q, w_dq = self._quantize(w)
                W_q_block[:, i] = w_q
                
                # 오차 계산 후 남은 column에 보정
                err = (w - w_dq) / h_inv_ii                         # [out_features]
                W_block[:, i+1:] -= err.unsqueeze(1) * H_inv_block[i, i+1:].unsqueeze(0)
                err_block[:, i] = err
            
            W_q[:, col_start:col_end] = W_q_block
            
            # 다음 block에 오차 전파
            W[:, col_end:] -= err_block @ H_inv[col_start:col_end, col_end:]
        
        return W_q

    
    def quantize(self):
        X_dict, W_dict = self.collect_input_n_weights()
        print(f"Calibration done. Total layers: {len(W_dict)}")
        
        for i, (name, module) in enumerate(self.model.named_modules()):
            if isinstance(module, nn.Linear):
                X = X_dict[name]
                W = W_dict[name]
                print(f"[{i}] Quantizing {name} | W: {W.shape} | X: {X.shape}", end=" ... ")
                H = self.Hessian(X)
                W_q = self._gptq(W, H)
                module.weight.data = W_q.to(module.weight.dtype)
                print(f"done")
        
        print("Quantization complete!")


quantizer = Quantizer(model, train_loader, bits=4)
quantizer.quantize()

KeyboardInterrupt: 